# LSTM Training Notebook for Indian Equity Predictor

**Purpose:** Train an LSTM model on NIFTY 50 stock data for use in the ensemble predictor.

**Instructions:**
1. Enable GPU: Settings → Accelerator → GPU T4 x2
2. Run all cells (takes ~15-20 minutes)
3. Download the 3 output files from the Output tab
4. Place them in `models/pretrained/` directory in your project

**Output files:**
- `lstm_weights.pth` — Model state dict
- `feature_scaler.pkl` — StandardScaler (fitted on training data)
- `model_config.json` — Model configuration

In [ ]:
!pip install -q yfinance ta scikit-learn torch

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import ta
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Download Data
We download 10 years of data for 20 large-cap NIFTY 50 stocks.

In [ ]:
TICKERS = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS',
    'HINDUNILVR.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'ITC.NS', 'KOTAKBANK.NS',
    'LT.NS', 'AXISBANK.NS', 'ASIANPAINT.NS', 'MARUTI.NS', 'TITAN.NS',
    'SUNPHARMA.NS', 'WIPRO.NS', 'HCLTECH.NS', 'TATAMOTORS.NS', 'POWERGRID.NS',
]

all_data = {}
for ticker in TICKERS:
    try:
        df = yf.Ticker(ticker).history(period='10y', auto_adjust=True)
        if len(df) > 500:
            all_data[ticker] = df
            print(f'  {ticker}: {len(df)} days')
    except Exception as e:
        print(f'  {ticker}: FAILED - {e}')

print(f'\nDownloaded {len(all_data)} stocks')

## 2. Feature Engineering
Same features as the local pipeline for consistency.

In [ ]:
def build_features(df):
    feat = pd.DataFrame(index=df.index)
    c = df['Close']
    h, l, v = df['High'], df['Low'], df['Volume']
    
    # Price features
    for w in [1, 5, 10, 20, 60]:
        feat[f'ret_{w}d'] = np.log(c / c.shift(w))
    for w in [10, 20, 60]:
        feat[f'vol_{w}d'] = c.pct_change().rolling(w).std()
    
    # Technical indicators
    feat['RSI_14'] = ta.momentum.rsi(c, window=14)
    stoch = ta.momentum.StochRSIIndicator(c, window=14)
    feat['StochRSI'] = stoch.stochrsi()
    macd = ta.trend.MACD(c)
    feat['MACD'] = macd.macd()
    feat['MACD_Signal'] = macd.macd_signal()
    feat['MACD_Hist'] = macd.macd_diff()
    bb = ta.volatility.BollingerBands(c, window=20, window_dev=2)
    feat['BB_PctB'] = bb.bollinger_pband()
    atr_ind = ta.volatility.AverageTrueRange(h, l, c, window=14)
    feat['ATR_14'] = atr_ind.average_true_range()
    adx = ta.trend.ADXIndicator(h, l, c, window=14)
    feat['ADX_14'] = adx.adx()
    feat['Williams_R'] = ta.momentum.williams_r(h, l, c, lbp=14)
    cci = ta.trend.CCIIndicator(h, l, c, window=20)
    feat['CCI_20'] = cci.cci()
    
    # Volume
    feat['Volume_Ratio'] = v / v.rolling(20).mean()
    feat['OBV_roc'] = ta.volume.on_balance_volume(c, v).pct_change(20)
    
    # Moving average positions
    sma50 = ta.trend.sma_indicator(c, window=50)
    sma200 = ta.trend.sma_indicator(c, window=200)
    feat['Price_vs_SMA50'] = (c - sma50) / sma50 * 100
    feat['Price_vs_SMA200'] = (c - sma200) / sma200 * 100
    
    # Time features
    dow = df.index.dayofweek
    feat['day_sin'] = np.sin(2 * np.pi * dow / 5)
    feat['day_cos'] = np.cos(2 * np.pi * dow / 5)
    month = df.index.month
    feat['month_sin'] = np.sin(2 * np.pi * month / 12)
    feat['month_cos'] = np.cos(2 * np.pi * month / 12)
    
    feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    feat.fillna(method='ffill', inplace=True)
    feat.fillna(0, inplace=True)
    return feat


def build_target(close, horizon=30, buy_thresh=0.05, sell_thresh=-0.05):
    future_ret = close.shift(-horizon) / close - 1.0
    target = pd.Series(1, index=close.index, dtype=int)
    target[future_ret > buy_thresh] = 2
    target[future_ret < sell_thresh] = 0
    target[future_ret.isna()] = -1
    return target

print(f'Feature count: {build_features(list(all_data.values())[0]).shape[1]}')

## 3. Prepare Dataset

In [ ]:
SEQ_LEN = 60
HORIZON = 30  # Train on 30-day prediction

all_X, all_y = [], []

for ticker, df in all_data.items():
    feat = build_features(df)
    target = build_target(df['Close'], horizon=HORIZON)
    
    valid = target != -1
    feat = feat[valid]
    target = target[valid]
    
    values = feat.values
    targets = target.values
    
    for i in range(SEQ_LEN, len(values)):
        all_X.append(values[i-SEQ_LEN:i])
        all_y.append(targets[i])

X = np.array(all_X, dtype=np.float32)
y = np.array(all_y, dtype=np.int64)
print(f'Dataset: {X.shape[0]} samples, {X.shape[2]} features, seq_len={SEQ_LEN}')
print(f'Class distribution: Sell={np.sum(y==0)}, Hold={np.sum(y==1)}, Buy={np.sum(y==2)}')

In [ ]:
# Scale features
n_samples, seq_len, n_features = X.shape
X_flat = X.reshape(-1, n_features)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_flat).reshape(n_samples, seq_len, n_features)

# Time-series split: 80% train, 20% test (no shuffling!)
split_idx = int(len(X_scaled) * 0.8)
X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## 4. Define LSTM Model

In [ ]:
class StockLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]
        out = self.dropout(last_hidden)
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

model = StockLSTM(input_size=n_features).to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## 5. Train

In [ ]:
# Class weights for imbalanced data
class_counts = np.bincount(y_train, minlength=3).astype(float)
class_weights = 1.0 / (class_counts + 1)
class_weights = class_weights / class_weights.sum() * 3
weights_tensor = torch.FloatTensor(class_weights).to(device)

train_ds = StockDataset(X_train, y_train)
test_ds  = StockDataset(X_test, y_test)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=256)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

EPOCHS = 50
best_loss = float('inf')
patience_counter = 0
PATIENCE = 10

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_dl:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in test_dl:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            out = model(X_batch)
            val_loss += criterion(out, y_batch).item()
    
    avg_train = train_loss / len(train_dl)
    avg_val   = val_loss / len(test_dl)
    scheduler.step(avg_val)
    
    if avg_val < best_loss:
        best_loss = avg_val
        torch.save(model.state_dict(), '/kaggle/working/lstm_weights.pth')
        patience_counter = 0
    else:
        patience_counter += 1
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | LR: {optimizer.param_groups[0]["lr"]:.6f}')
    
    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1}')
        break

print(f'Best validation loss: {best_loss:.4f}')

## 6. Evaluate

In [ ]:
model.load_state_dict(torch.load('/kaggle/working/lstm_weights.pth'))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for X_batch, y_batch in test_dl:
        X_batch = X_batch.to(device)
        out = model(X_batch)
        preds = torch.argmax(out, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y_batch.numpy())

print('Classification Report:')
print(classification_report(all_true, all_preds, target_names=['Sell', 'Hold', 'Buy']))
print('Confusion Matrix:')
print(confusion_matrix(all_true, all_preds))

## 7. Export

In [ ]:
# Save scaler
joblib.dump(scaler, '/kaggle/working/feature_scaler.pkl')

# Save config
config = {
    'n_features': n_features,
    'seq_len': SEQ_LEN,
    'hidden_size': 128,
    'num_layers': 2,
    'dropout': 0.3,
    'num_classes': 3,
    'horizon': HORIZON,
}
with open('/kaggle/working/model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Files saved to /kaggle/working/')
print('Download: lstm_weights.pth, feature_scaler.pkl, model_config.json')
print('Place in models/pretrained/ directory')